# CSCI6032 Homework 6

**Instructions**

* **Insert all code, plots, results, and discussion** into this Jupyter Notebook.
* Your homework should be submitted as a **single Jupyter Notebook** (.ipynb file) to **Blackboard**.
* While working, you can use Visual Studio Code, Jupyter Notebook or Google Colab.  If using Google Colab, export it as a Jupyter Notebook (.ipynb).

You can answer mathematical questions either by:
* using LaTeX in a markdown cell, or
* handwriting your answer, then photographing or scanning it and using an AI/LLM to convert it into markdown and LaTeX, checking the conversion for accuracy before submitting.


---
## Problem 1: Entropy and Run Length Encoding

Given a source that outputs a sequence of zeros and ones.  Each zero or one is called a symbol.  
In this problem, each symbol is independent of the others.  The probability of a 1 is given by
$p$ and thus the probability of a zero is $1-p$.

**(a)** Compute the Shannon entropy for $p=0.3$ (3 points)

**Answer (a)**

**(b)** Plot the Shannon entropy in bits for many values of $p$ in the interval $[0,1]$. (3 points)

In [ ]:
# HERE

**(c)** Consecutive occurrences of the same symbol are called runs. `000001` is a run
of 5 zeros followed by a 1.  *Run length encoding* 
is sometimes used when the input data is dominated by long runs of the same symbol.
`AAABBBBBEEEE` would be encoded as `3A5B4E`. 

*Zero run length encoding* is a variant typically used with binary sequences
in which the dominant symbol is zero.  We allocate a fixed length field $k$
bits wide to denote the run length of consecutive zeros before a 1.  If all bits
in the field are ones, this is a reserved value indicating an *incomplete run*.

An *incomplete run* occurs when a run of zeros reaches the maximum length
representable by the $k$-bit field, which is $2^k-1$, and continues with
additional zeros without an intervening 1.  In this case, the next field 
encodes the length of the continued run.

For example, if $k=4$, the zero run length encoding creates the given
outputs for the input sequences shown in Table 1.

$$\text{Table 1}$$

| input           |  output      | compression$\dagger$ |  explanation                            |
|-----------------|--------------|-------------|--------------------------------------------------|
| 1               | 0000         | 1/4         | zero zeroes then 1                               |
| 01              | 0001         | 1/2         | one zero then 1                                  |
| 11              | 00000000     | 1/4         | 0 zeroes then 1, 0 zeros then 1                  |
| 0000000001      | 1001         | 10/4=2.5    | $1001_2=9$, 9 zeroes then 1.                     |
| 0010001         | 00100011     | 7/8         | $0010_2=2$ zeroes, $0011_2=3$ zeroes.            |
| 32 zeros then 1 | 111111110010 | 33/12=2.75  | $15(1111_2) + 15(1111_2) + 2 = 32$ zeroes then 1 |


$$\dagger \text{ compression ratio is } \frac{\text{uncompressed size}}{\text{compressed size}}$$

In the last row we reduce 33 input symbols to 12 output symbols.  This has a compression
ratio of $33/12 = 2.75$

For $p=0.003$, (i.e., the probability of a 1 is 0.3%), implement zero run length encoding.  It should pass the provided tests. (7 points)  

In [ ]:
import io
import unittest
import random

def vassert(exp):
    if not exp:
        raise ValueError()

    
# I chose to follow the design pattern used by zlib and zstandard
# which allows compression parameter to passed as an optional argument
# to a compress method.
#
# Addendum: as described in the original problem, this encoding
# doesn't handle termination properly if the file ends with a zero.
# See Problem 1(h) for a version that does.
def compress(data: bytes, k: int = 4) -> bytes:
    vassert(k > 0)
    # HEREHEREHERE
    return buffer

def decompress(data: bytes, k: int = 4) -> bytes:
    vassert(k > 0)
    # HEREHEREHERE
    return buffer        

def create_buffer(size, percent_ones):
    num_ones = int((percent_ones / 100) * (size - 1))  # Reserve 1 for the terminating '1'
    num_zeros = size - 1 - num_ones  # Remaining space for zeros
    buffer_list = ['1'] * num_ones + ['0'] * num_zeros

    # Shuffle to randomize the placement of '1's and '0's
    random.shuffle(buffer_list)

    # Append a terminating '1'
    buffer_list.append('1')

    # Join the list into a byte string of ASCII characters
    buffer = ''.join(buffer_list).encode('ascii')
    return buffer

class TestCompress(unittest.TestCase):
    def test_edge_cases(self):
        try:
            compress(None)
            self.assertTrue(False)
        except:
            pass

        try:
            compress(b"010", -1)
            self.assertTrue(False)
        except ValueError:
            pass

        buf = compress(b"")
        self.assertEqual(len(buf), 0)

    def test_simple_cases(self):
        # This does not handle terminating edge condition well. An ending 0000
        buf = compress(b"1")
        self.assertEqual(buf, b"0000")

        buf = compress(b"01")
        self.assertEqual(buf, b"0001")

        buf = compress(b"11")
        self.assertEqual(buf, b"00000000")

    def test_more_complicated_cases(self):
        buf = compress(b"0000000001")
        self.assertEqual(buf, b"1001")
        
        buf = compress(b"0"*32 + b"1")
        self.assertEqual(buf, b"111111110010")

    def test_decompression(self):
        self.assertEqual(decompress(compress(b"1")), b"1")
        self.assertEqual(decompress(compress(b"01")), b"01")
        self.assertEqual(decompress(compress(b"11")), b"11")

    def test_complicated_decompression(self):
        self.assertEqual(decompress(compress(b"0000000001")), b"0000000001")
        self.assertEqual(decompress(compress(b"0"*32 + b"1")), b"0"*32 + b"1")

    def test_3_percent(self):
        buf = create_buffer(size=1000, percent_ones=3)
        self.assertEqual(decompress(compress(buf)), buf)

    def test_p1c(self):
        with open("p1c.txt", "rb") as fp:
            buf = fp.read()
        self.assertEqual(decompress(compress(buf)), buf)
        
unittest.main(argv=[''], verbosity=2, exit=False)       

The file `p1c.txt.gz` is in this homework's folder on GitHub.  Decompress it using `gunzip` before using it.
You will need `p1c_zero_end.txt.gz` in part (h); decompress that one too.

In [ ]:
import os
import glob
print("working directory:", os.getcwd()) 
print('glob("*.txt"):', glob.glob("*.txt"))
assert os.path.exists("p1c.txt")

**(d)** Use your compression algorithm to compress the data file `p1c.txt` that uses ASCII 0
and 1 to denote zeros and ones.  Output the compressed text file with name `p1c_k4.txt`.
The output file should use ASCII zeros and ones, but compress the data using zero run
length encoding with k=4. (3 points)

In [ ]:
# HERE

**(e)** Plot the compression ratio of `p1c.txt` using the zero run length encoder 
built in (c) as a function of $k$ for $k = 4, 5, \ldots, 16$.  Put $k$ on the x-axis and
compression ratio on the y-axis. (4 points)

In [ ]:
# HERE

**(f)** Plot the length of the output file as $k$ varies from 4 to 16. Plot a horizontal
line at $n \times H(x)$, where $n$ is the length of the input file `p1c.txt` and $H(X)$ for
$p=0.003$.  Plot a second horizontal line for the length of the
file `p1c.txt.gz`. The value $n \times H(X)$ represents the theoretical lower bound 
achievable for lossless compression.  NOTE: To make this fair, consider the length in 
units of symbols.  Since your symbols are ascii characters, multiply the length of 
the gzip file by 8. (4 points)


In [ ]:
# HERE

**(g)** How does $n \times H(X)$ from (f) compare to the length of the compressed files
using your zero run-length encoding vs. gzip? (3 points)

In [ ]:
# HERE

**(h)** What happens if the file ends on a zero?  This encoding is only defined for files that end 
with a 1.  If we modify the encoding so that the last field in the file does not denote a run length but 
rather is either a 1 or 0 denoting whether the file ends in a complete (1) or incomplete (0) run.   
If the file ended on a complete run that means that the last input symbol was a 1.

$$\text{Table 2}$$
| input            |  output          | compression$\dagger$ |  explanation                                     |
|-----------------|------------------|--------------|-----------------------------------------------------------|
| 1               | 00000001         | 1/8          | zero zeroes then 1. Ending 0001 = complete run            |
| 01              | 00010001         | 1/4          | one zero then 1. Ending 0001 = complete run               |
| 11              | 000000000001     | 1/6          | 0 zeroes then 1, 0 zeros then 1. 0001 = complete run      |
| 0000000001      | 10010001         | 10/8=1.25    | $1001_2=9$, 9 zeroes then 1.  0001 = complete run         |
| 0010001         | 001000110001     | 7/12         | $0010_2=2$ zeroes, $0011_2=3$ zeroes. 0001 = complete run |
| 32 zeros then 1 | 1111111100100001 | 33/16=2.0625 | $15(1111_2) + 15(1111_2) + 2 = 32$ zeroes then 1          |
| 32 zeros        | 1111111100100000 | 32/16=2      | ending 0000 = incomplete run                              |
| 0               | 00010000         | 1/8          | one zero. Ending 0000 = incomplete run                    |


For $p=0.003$, (i.e., the probability of a 1 is 0.3%), implement this modified zero run length encoding 
as `compress2` and `decompress2`.

It must handle `p1c_zero_end.txt.gz`, which is `p1c.txt.gz` with its final 1 removed so that it ends with a zero. (7 points)

In [ ]:
import io
import sys
import unittest


def vassert(exp):
    if not exp:
        raise ValueError()

    
# I chose to follow the design pattern used by zlib and zstandard
# which allows compression parameter to passed as an optional argument
# to a compress method.
#
# Addendum: as described in the original problem, this encoding
# doesn't handle termination properly if the file ends with a zero.
# To handle this case, I am adding one additional behavior that is
# specific to the last k-width field in the file.  The last field 
# right before the EOF encodes whether the previous run ended in a 
# one or zero.  If the last value is a one then the last run ends 
# with a one.  If the last value is a zero then the last run is an 
#incomplete run.  
def compress2(data: bytes, k: int = 4) -> bytes:
    # HEREHEREHERE
    return buffer        

def decompress2(data: bytes, k: int = 4) -> bytes:
    vassert(k > 0)
    # HEREHEREHERE
    return buffer

class TestCompress2(unittest.TestCase):
    def test_edge_cases(self):
        try:
            compress2(None)
            self.assertTrue(False)
        except:
            pass

        try:
            compress2(b"010", -1)
            self.assertTrue(False)
        except ValueError:
            pass

        buf = compress2(b"")
        self.assertEqual(len(buf), 0)

    def test_simple_cases(self):
        # This does not handle terminating edge condtion well. An ending 0000
        buf = compress2(b"1")
        self.assertEqual(buf, b"00000001") # 0000 for 1, 0001 to denote complete run.

        buf = compress2(b"0")
        self.assertEqual(buf, b"00010000") # 0001 for a run of 1 zero. 0000 to denote an incomplete run.

        buf = compress2(b"01")
        self.assertEqual(buf, b"00010001")

        buf = compress2(b"11")
        self.assertEqual(buf, b"000000000001")

        buf = compress2(b"00", k=3)
        self.assertEqual(buf, b"010000")

    def test_decompress2(self):
        self.assertEqual(decompress2(compress2(b"1")), b"1")
        self.assertEqual(decompress2(compress2(b"01")), b"01")
        self.assertEqual(decompress2(compress2(b"11")), b"11")
        
    def test_p1c_zero_end(self):
        with open("p1c_zero_end.txt", "rb") as fp:
            buf = fp.read()
        self.assertEqual(decompress2(compress2(buf)), buf)


unittest.main(argv=[''], verbosity=2, exit=False)  


**(i)** Show the same plot as in **(f)** but include the results for both  `compress2` and `compress` as a sanity check to ensure that the 
additional field appended to the end doesn't otherwise affect efficiency of the compression.   The two curves should almost entirely coincide. (3 points)

In [ ]:
#HERE


---
## Problem 2: Cross-Entropy Loss

In classification problems, our classifier estimates the posterior 
probability $P(\omega_i | \bar{x})$ for every class.
This can be expressed as a vector of probabilities.
For example, if we are in the region of ambiguity for three different classes
such that $P(\omega_1 | \bar{x}) = 0.3$, $P(\omega_2 | \bar{x}) = 0.5$
and $P(\omega_3 | \bar{x}) = 0.2$.  The classifier doesn't know the 
true posterior probabilities so it estimates them.  

Let $\hat{P}(\omega_j | \bar{x})$ denote the estimated posterior probability
of class $j$ given feature vector $\bar{x}$.  The estimated posterior
probability is called the *predicted probability*.

Let $y$ denote the true class.  When discussing training sets, in
previous lectures we introduced $y^{(i)}$ as a scalar denoting the 
true class for the $i$th observation $(\mathbf{x}^{(i)}, y^{(i)})$.
We continue to use $y^{(i)}$ notation in the context of training sets.

Let $\hat{y}_j$ denotes the predicted probability for class $j$.
$\hat{y}_j$ estimates the posterior probability. Meaning

$$\hat{y}_j = \hat{P}(\omega_j | \bar{x}) \hspace{2in} (2.1)$$

Let $\mathbf{\hat{y}}$ denote the column vector of the predicted 
probabilities.

$$
\mathbf{\hat{y}} = \begin{bmatrix}
\hat{P}(y = 1 | \mathbf{x}) \\
\hat{P}(y = 2 | \mathbf{x}) \\
\vdots \\
\hat{P}(y = K | \mathbf{x})
\end{bmatrix}
$$

An accurate classifier would closely model the posterior
probabilities such that

$$
\mathbf{\hat{y}} \approx \begin{bmatrix}
P(y = 1 | \mathbf{x}) \\
P(y = 2 | \mathbf{x}) \\
\vdots \\
P(y = K | \mathbf{x})
\end{bmatrix}
$$

Thus for the example probabilities $P(\omega_1 | \bar{x}) = 0.3$, $P(\omega_2 | \bar{x}) = 0.5$
and $P(\omega_3 | \bar{x}) = 0.2$, $\mathbf{\hat{y}}$ becomes

$$
\mathbf{\hat{y}} \approx \begin{bmatrix}
0.3 \\
0.5 \\
0.2
\end{bmatrix}
$$

Cross-entropy is a widely used loss function in
machine learning, especially for multi-class classification tasks, 
because it measures a notion of distance between the predicted probability
distribution and the true distribution.

The cross-entropy between two probability distributions $P$ and $Q$
over the same set of events is defined as:

$$H(P, Q) = - \sum_{j} P(j) \log Q(j)$$

For classification we replace $P$ with a distribution that places
100% of the probabilty on the correct class, and we replace
$Q$ with $\mathbf{\hat{y}}$.

In other words
$P(y=k) = 1$ where $k$ is the true class and for all $j\neq k$, 
$P(y = j) = 0$.  This probability assignment can be represented using
an indicator function.

An indicator function is defined by

$$
\mathbb{1}\{ y = k \} =
\begin{cases}
1 & \text{if } y = k \\
0 & \text{if } y \neq k
\end{cases}
$$

Using an indicator function, cross entropy becomes

$$H(\mathbb{1}\{ y = k \}, \mathbf{\hat{y}}) = - \sum_{j=1}^K \mathbb{1}\{ y = j \} \log \mathbf{\hat{y}}_j \hspace{2in} (2.2)$$





Indicator functions are commonly used, but I prefer a less heavy notation using
what is called a *one-hot vector*.  As with indicator functions, one-hot vectors
are also heavily used in machine learning.

Let $\mathbf{y}(k)$ denote the one-hot vector wherein $y=k$ corresponds to the 
$k$th element being 1, e.g.,

$$
\mathbf{y}(k) = \begin{bmatrix}
0 \\
0 \\
\vdots \\
0 \\
1 \\
0 \\
\vdots \\
0
\end{bmatrix}
$$

where the $1$ above appears in the $k$th row of the column vector.

For brevity, we may simply refer to the one-hot vector as $\mathbf{y}$.  Note that $y$ is a 
scalar denoting the true class, and $\mathbf{y}$ is the one-hot vector with 100% of the
probability at the index of the true class.

Using matrix notation, (2.2) can be restated as 

$$H(\mathbf{y}, \mathbf{\hat{y}}) = - \mathbf{y}^T \log \mathbf{\hat{y}} \hspace{2in} (2.3)$$

In classification problems, the loss function is often denoted by a calligraphic L, i.e.,
$\mathcal{L}$, thus when we use cross-entropy $H$ as a loss function, we rewrite (2.3) as

$$\mathcal{L}(\mathbf{y}, \mathbf{\hat{y}}) = - \mathbf{y}^T \log \mathbf{\hat{y}} \hspace{2in} (2.4)$$

**(a)** Suppose we have a four-class classification problem 
$K = 4$, and for a particular data point, the true label is $y=3$. Write the one-hot vector $\mathbf{y}$ for y=3. (3 points)


**Answer 2(a):**

HERE

**(b)** The predicted probability vector $\mathbf{\hat{y}}$ for this data point
output from our model is:

$$
\mathbf{\hat{y}} = \begin{bmatrix}
0.1 \\
0.2 \\
0.6 \\
0.1
\end{bmatrix}
$$

Compute the cross-entropy loss $\mathcal{L}$ when y = 3.  Assume $\log$ base 2. (3 points)

**Answer 2(b)**

HERE

**(c)** If the true class were $y=1$, what would $\mathcal{L}$ equal? (3 points)

**Answer 2(c)**

HERE

**(d)** If the predicted probability vector is itself a one-hot vector where $\hat{y}_k = 1$ and the true class is $y=k$, what is the loss $\mathcal{L}$? (3 points) 

**Answer 2(d)**

HERE

**(e)** The classifier is certain that $\mathbf{x}$ corresponds to class 4, i.e., $\hat{y}_4 = 1$, but the true class is $y=3$.  What happens to $\mathcal{L}$.  Aside: this problem can be avoided by clipping the predicted probabilities such that 
$\hat{y}_k \in [\epsilon, 1 - \epsilon]$ where $\epsilon$  is a small value like $10^{-8}$, to avoid $\log(0)$. (3 points)


**Answer 2(e)**

HERE


---
## Problem 3: Information Gain for a Spam Detector

Let X be the dataset at a node and $Y \in \text{labels}$ the class label.
For a split $\sigma$ (e.g., a threshold on a feature or a categorical partition):

$$\mathrm{IG}(X,\sigma) \;=\; H(Y)\;-\;H(Y\mid \sigma),$$

where

$$H(Y) \;=\; -\sum_{y} P(Y=y)\log_2 P(Y=y), \qquad
H(Y\mid \sigma)\;=\;\sum_{b} P(B_b)\, H\!\left(Y\,\big|\,B_b\right)
\;=\;\sum_{b} \frac{|X_b|}{|X|}\,H(Y_{X_b}).$$

Here $B_b$ indexes the branches (children) produced by $\sigma$, $X_b$ is
the subset of $X$ in branch $b$, and $Y_{X_b}$ are the labels restricted
to $X_b$. Use base-2 logs.  You can see examples in the lecture notes on decision trees.

Consider the case of an email spam detector:
`Y = Spam?` (`+` = spam, `-` = not spam)

$X$ has 14 items with class counts $(n_{+}, n_{-}) = (9,5)$.


| id | HasAttachment | SenderDomainType | Y |
|----|----------------|------------------|---|
|  1 | 0 | free      | + |
|  2 | 1 | free      | + |
|  3 | 0 | free      | - |
|  4 | 1 | free      | - |
|  5 | 0 | corporate | + |
|  6 | 0 | corporate | + |
|  7 | 1 | corporate | + |
|  8 | 0 | corporate | + |
|  9 | 1 | corporate | - |
| 10 | 0 | unknown   | + |
| 11 | 0 | unknown   | + |
| 12 | 0 | unknown   | - |
| 13 | 1 | unknown   | + |
| 14 | 1 | unknown   | - |


$H(Y) = -(\tfrac{9}{14} \log_2 \tfrac{9}{14} + \tfrac{5}{14} \log_2 \tfrac{5}{14}) \approx 0.9403$

(a) Compute the entropy $H(Y|\text{HasAttachment})$ and IG for the `HasAttachment` feature. (4 points)

(b) There are three ways to group sender domain types into a single binary decision: 
- $\sigma_{\texttt{free}}$:“In free?” vs “corporate or unknown”
- $\sigma_{\texttt{corporate}}$: “In corporate?” vs “free or unknown”
- $\sigma_{\texttt{unknown}}$: “In unknown?” vs “free or corporate”

For each decision $\sigma$ above, compute $H(Y\mid \sigma)$ and 
$\mathrm{IG}(X,\sigma)$. Report all three values. (6 points)

(c) Draw the binary decision tree for the feature-splits that maximizes IG at each interior
node. Include the class posterior probabilities and class label for each leaf.  Stop
when no split yields positive gain. (6 points)



---
## Problem 4: A Decision Tree from Scratch

Implement a binary decision tree classifier from scratch using only numpy and standard python libraries
(no pandas, scikit-learn, etc.).

We provide two CSV files in this homework's folder on GitHub:
- `spam_train.csv`
- `spam_test.csv`

`spam_train.csv` has 500 emails and `spam_test.csv` has 300.  Each row is an email with an `id` (not a feature),
five features, and a binary label:

- `HasAttachment` ∈ {0,1}
- `SenderDomainType` ∈ {free, corporate, unknown}
- `NumLinks` (float)
- `SubjectLength` (int, number of characters)
- `IsWeekend` ∈ {0,1}
- `Y` ∈ {0,1} where 1 = spam, 0 = not spam

(a) Implement the following API:

- `class DecisionTree:`
  - `__init__(self, max_depth=None, min_leaf_size=1, m_try=None, random_state=None)`
  - `fit(X, y)`
  - `predict(X) -> np.ndarray[int]`

`m_try` is the number of features to consider at each split.  They are chosen uniformly at random, without
replacement, and re-chosen at every node.  `None` means consider all features, which is an ordinary decision tree.
`random_state` seeds that random choice.  You will need `m_try` in Problem 5. (9 points)

(b) train a decision tree using `spam_train.csv` with default (infinite)
`max_depth` and default `min_leaf_size` of 1. (3 points)

(c) measure the accuracy of your decision tree using `spam_test.csv`.  The labels are noisy, so even the best
possible classifier makes mistakes on this data. (3 points)

(d) Compare your accuracy to that of sklearn's tree. (2 points)

    from sklearn import tree

Note: You can only use sklearn for (d).

**Tie-break:** if IG ties, prefer based on alphabetical order of the feature name then minimum split threshold or first category name.



---
## Problem 5: A Random Forest from Scratch

Implement a random forest classifier from scratch using only numpy and standard python libraries.

#### Requirements

- **Bootstrap sampling:** For each tree, draw a bootstrap sample *with replacement* from the training set. The bootstrap sample should have the same size as the training set.
- **Feature subsampling (m_try):**
  - Classification default: `m_try = max(1, floor(sqrt(p)))`, where `p` = number of features.
  - At each split, consider only a uniformly random subset of `m_try` features (re-sampled per node).
- **Number of trees:** Default `T = 50` (see `n_trees` parameter).
- **Aggregation:**
  - Class prediction = **majority vote** across trees.
  - Class probability = **mean of leaf probabilities** for class 1 across trees.

- **Reproducibility:** Accept `random_state` to seed RNG; use `numpy.random.Generator`.

(a) Implement the following API:

- `class RandomForest:`
  - `__init__(self, n_trees=50, m_try=None, max_depth=None, min_leaf_size=1, random_state=None)`
  - `fit(X, y)`
  - `predict(X) -> np.ndarray[int]` (majority vote)
  - `predict_proba(X) -> np.ndarray[float]` (the probability of class 1 for each row, the mean of the leaf probabilities)

Build each tree with your `DecisionTree` from Problem 4, passing it `m_try`. (9 points)

(b) train a random forest using `spam_train.csv` (2 points)

(c) measure the accuracy of your random forest classifier using `spam_test.csv` (2 points)

(d) Compare the performance of your implementation to sklearn's `RandomForestClassifier` (2 points):

    from sklearn.ensemble import RandomForestClassifier

Note: You can only use sklearn for (d).
